# Quickstart: ECMWF IFS ENS forecast, 15 day, 0.25 degree - dynamical.org Zarr
A brief introduction to the ECMWF IFS Ensemble forecast dataset transformed into an analysis-ready, cloud-optimized format by dynamical.org.

Dataset documentation: https://dynamical.org/catalog/ecmwf-ifs-ens-forecast-15-day-0-25-degree/


In [ ]:
# If running locally, follow README.md for simple dependency installation.
# If using Google Colab, run this cell and then restart the notebook.
%pip install "xarray[complete]>=2025.1.2" "zarr>=3.0.8" requests aiohttp

In [ ]:
import numpy as np
import xarray as xr

In [ ]:
ds = xr.open_zarr("https://data.dynamical.org/ecmwf/ifs-ens/forecast-15-day-0-25-degree/latest.zarr?email=optional@email.com", decode_timedelta=True, chunks=None)
ds

## Temporary: checking out ingested_forecast_length

In [ ]:
ds.coords['ingested_forecast_length']

In [ ]:
# Plot expected vs ingested forecast length over time
fig, ax = plt.subplots(1,1, figsize=(12, 6))
expected_length_hours = [td / np.timedelta64(1, 'h') for td in ds.coords['expected_forecast_length'].values]
ingested_length_hours = [td / np.timedelta64(1, 'h') for td in ds.coords['ingested_forecast_length'].values]
ax.plot(ds.init_time.values, ingested_length_hours, label="Ingested forecast length")
ax.plot(ds.init_time.values, expected_length_hours, label="Expected forecast length", linestyle='dashed')

ax.set_title("Expected vs Ingested Forecast Length")
ax.grid(True, alpha=0.3)
ax.legend()

## Resuming with the regularly scheduled programming of plots straight up stolen from the gefs notebook (note: only early datetimes for now!)

# Plot the ensemble traces of the 2024-04-03 forecast at a point on the earth
plot_ds = ds.sel(init_time="2024-04-03T00", latitude=-23.5, longitude=-46.6, method="nearest")  # São Paulo, Brazil
_ = plot_ds["temperature_2m"].plot(x="valid_time", hue="ensemble_member", figsize=(12, 8))

In [ ]:
# Plot a summary of the ensemble distribution using quantiles
plot_ds = ds.sel(init_time="2024-05-01T00", latitude=0, longitude=0)
(
    plot_ds["temperature_2m"]
    .quantile([0.05, 0.25, 0.5, 0.75, 0.95], dim="ensemble_member")
    .plot(x="valid_time", hue="quantile")
)

In [ ]:
# The following larger area examples run faster using dask which happens by default if you omit chunks=None
ds = xr.open_zarr("https://data.dynamical.org/ecmwf/ifs-ens/forecast-15-day-0-25-degree/latest.zarr?email=optional@email.com", decode_timedelta=True)

In [ ]:
# Calculate a quantile across ensemble members and display the result as a map
(
    ds["temperature_2m"]
    .sel(init_time="2024-05-01T00")
    .sel(lead_time="7d")
    .sel(latitude=slice(70, 20), longitude=slice(0, 50))
    .quantile(0.25, dim="ensemble_member") # 25% chance it gets colder than this
    .plot()
)

In [ ]:
# Highlight areas of uncertainty in temperature forecast over the first 7 days of the first forecast

import matplotlib.pyplot as plt

plot_ds = (
    ds.sel(init_time="2024-04-03T00")
    .sel(latitude=slice(-8, -46), longitude=slice(112, 154))  # Australia
    .sel(lead_time=slice("0h", "6d")).mean(dim="lead_time")  # Average the first week of the forecast
)

# Standard deviation across ensemble members to highlight regions of forecast uncertainty
plot_ds["temperature_2m"].std(dim="ensemble_member").plot()
plt.title(f"Ensemble standard deviation 2 meter temperature [{ds['temperature_2m'].attrs['units']}]")

plt.tight_layout()

## Messing with some additional plots

In [ ]:
# Plot the ensemble traces of the 2024-04-03 forecast at a point on the earth
plot_ds = ds.sel(init_time="2024-04-03T00", latitude=37.75, longitude=-122.5, method="nearest")  # San Francisco, USA
fig, ax = plt.subplots(1,1, figsize=(12,8))
plot_ds.sel(ensemble_member=slice(1,51))["temperature_2m"].plot(x="valid_time", hue="ensemble_member", ax=ax, alpha=0.4, label="Perturbed ensemble members");
plot_ds.sel(ensemble_member=0)["temperature_2m"].plot(x="valid_time", ax=ax, color="black", label="Control ensemble member");

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[-2:], labels[-2:])
ax.set_title("Control + Perturbed members for 2024-04-03's temperature forecast in San Francisco");